# 06 · X2C-GHF with DIIS

X2C-GHF 在 GHF 的 spin-orbital AO 基上替换一电子 Hamiltonian：

$$
\{\chi_\mu\alpha,\chi_\mu\beta\}
$$

非相对论 GHF 的 $H^{\mathrm{core}}$ 是 real block diagonal；X2C-GHF 的 $H^{\mathrm{X2C}}$ 包含 spin-dependent 项，因此是 complex Hermitian 矩阵。二电子 Coulomb/exchange 部分仍按 GHF 的 spin block 结构构造。

本 notebook 参考 PySCF `pyscf/x2c/x2c.py` 中 `SpinOrbitalX2CHelper` 与 `x2c1e_ghf` 的实现。

---

## 1 · Spin-Orbital X2C 一电子 Hamiltonian

PySCF 的 spin-orbital X2C-GHF 使用 Gaussian spin-orbital basis。先构造

$$
\mathbf{S}=\begin{pmatrix}S&0\\0&S\end{pmatrix},\quad
\mathbf{T}=\begin{pmatrix}T&0\\0&T\end{pmatrix},\quad
\mathbf{V}=\begin{pmatrix}V&0\\0&V\end{pmatrix}
$$

spin-dependent potential 项写成

$$
\mathbf{W} = \boldsymbol{\sigma}\cdot \mathbf{W}_{\mathrm{spnucsp}}
$$

然后解 large/small component 的 generalized eigenvalue problem：

$$
\begin{pmatrix}
V & T \\
T & W/(4c^2)-T
\end{pmatrix}
\begin{pmatrix}C_L\\C_S\end{pmatrix}
=
\begin{pmatrix}
S & 0\\
0 & T/(2c^2)
\end{pmatrix}
\begin{pmatrix}C_L\\C_S\end{pmatrix}E
$$

对正能解定义

$$
X = C_S C_L^{-1}
$$

再用 Foldy-Wouthuysen 形式得到二分量 Hamiltonian：

$$
\tilde{S}=S+\frac{1}{2c^2}X^\dagger T X
$$

$$
H_{\mathrm{FW}}
=
V + TX + X^\dagger T - X^\dagger T X
+ \frac{1}{4c^2}X^\dagger W X
$$

最后用 renormalization matrix $R$ 变回普通 AO overlap 度量：

$$
H_{\mathrm{X2C}} = R^\dagger H_{\mathrm{FW}} R,\qquad
R^\dagger \tilde{S} R = S
$$

PySCF 源码里 `_x2c1e_get_hcore` 采用了等价但更稳定的变换形式；这里显式写出 $X$ 和 $R$，便于看清物理结构。

---

## 2 · 分子设定与积分

这里继续使用 OH radical doublet。`XUNCONTRACT=True` 对应 PySCF X2C helper 的默认行为：在 uncontracted basis 中构造 X2C hcore，再收缩回原始 AO 基。

In [1]:
from functools import reduce

from pyscf import gto, lib, scf
from pyscf.gto import mole
from pyscf.x2c import x2c
import numpy as np
from scipy.linalg import block_diag, eigh

# -------- 分子定义: OH radical / cc-pVDZ --------
mol = gto.M(
    atom="O 0 0 0; H 0 0 0.9697",
    basis="cc-pvdz",
    spin=1,
    verbose=0,
)

XUNCONTRACT = True

nao = mol.nao_nr()
nso = 2 * nao
nelec = mol.nelectron
nalpha, nbeta = mol.nelec
E_nn = mol.energy_nuc()
c_light = lib.param.LIGHT_SPEED

S_nr = mol.intor_symmetric("int1e_ovlp")
T_nr = mol.intor_symmetric("int1e_kin")
V_nr = mol.intor_symmetric("int1e_nuc")
H_core_nr = T_nr + V_nr
eri = mol.intor("int2e")

S_so = block_diag(S_nr, S_nr)

print(f"nao = {nao}, nso = {nso}, nelec = {nelec}")
print(f"nalpha = {nalpha}, nbeta = {nbeta}")
print(f"XUNCONTRACT = {XUNCONTRACT}")
print(f"S_so shape = {S_so.shape}")
print(f"eri shape = {eri.shape} ({eri.size:,} elements)")
print(f"E_nn = {E_nn:.10f} Hartree")

nao = 19, nso = 38, nelec = 9
nalpha = 5, nbeta = 4
XUNCONTRACT = True
S_so shape = (38, 38)
eri shape = (19, 19, 19, 19) (130,321 elements)
E_nn = 4.3656983473 Hartree


---

## 3 · X2C hcore 构造

In [2]:
def block_diag_spin(mat):
    # [A 0; 0 A] in alpha-all / beta-all spin-orbital ordering.
    return block_diag(mat, mat)


def sigma_dot(mat):
    # PySCF x2c._sigma_dot: contract int1e_spnucsp with Pauli matrices.
    quaternion = np.vstack([1j * lib.PauliMatrices, np.eye(2)[None, :, :]])
    nao = mat.shape[-1] * 2
    return np.einsum("sxy,spq->xpyq", quaternion, mat, optimize=True).reshape(nao, nao)


def get_xmol(mol, xuncontract=True):
    # Minimal version of PySCF X2CHelperBase.get_xmol for ordinary GTO bases.
    if xuncontract:
        if not np.all(mol._bas[:, mole.KAPPA_OF] == 0):
            raise NotImplementedError("spinor decontraction for nonzero kappa is not implemented here")
        return mol.decontract_basis(atoms=xuncontract, aggregate=True)
    return mol, None


def get_rmat(S, S_nesc, tol=1e-14):
    # R satisfies R^dagger S_nesc R = S.
    eig, vec = np.linalg.eigh(S)
    keep = eig > tol
    vec = vec[:, keep]
    s_sqrt = np.sqrt(eig[keep])
    s_invsqrt = 1.0 / s_sqrt

    S_nesc_orth = reduce(np.dot, (vec.conj().T, S_nesc, vec))
    mid = np.einsum("i,ij,j->ij", s_invsqrt, S_nesc_orth, s_invsqrt, optimize=True)

    eig_mid, vec_mid = np.linalg.eigh(mid)
    keep_mid = eig_mid > tol
    vec_mid = vec_mid[:, keep_mid]
    mid_invsqrt = np.dot(vec_mid / np.sqrt(eig_mid[keep_mid]), vec_mid.conj().T)

    R_orth = np.einsum("i,ij,j->ij", s_invsqrt, mid_invsqrt, s_sqrt, optimize=True)
    return reduce(np.dot, (vec, R_orth, vec.conj().T))


def x2c_xmatrix(T, V, W, S, c):
    n = S.shape[0]
    h = np.zeros((2 * n, 2 * n), dtype=np.result_type(T, V, W, S))
    m = np.zeros_like(h)

    h[:n, :n] = V
    h[:n, n:] = T
    h[n:, :n] = T
    h[n:, n:] = W * (0.25 / c**2) - T

    m[:n, :n] = S
    m[n:, n:] = T * (0.5 / c**2)

    e, coeff = eigh(h, m)
    C_large = coeff[:n, n:]
    C_small = coeff[n:, n:]
    return np.linalg.solve(C_large.T, C_small.T).T


def x2c_hcore_fw(T, V, W, S, X, c):
    S_nesc = S + reduce(np.dot, (X.conj().T, T, X)) * (0.5 / c**2)
    TX = T @ X
    H_fw = (
        V
        + TX
        + TX.conj().T
        - X.conj().T @ TX
        + reduce(np.dot, (X.conj().T, W, X)) * (0.25 / c**2)
    )
    R = get_rmat(S, S_nesc)
    return reduce(np.dot, (R.conj().T, H_fw, R))


def get_x2c_hcore(mol, xuncontract=True):
    xmol, contr_coeff = get_xmol(mol, xuncontract=xuncontract)

    T = block_diag_spin(xmol.intor_symmetric("int1e_kin"))
    V = block_diag_spin(xmol.intor_symmetric("int1e_nuc"))
    S = block_diag_spin(xmol.intor_symmetric("int1e_ovlp"))
    W = sigma_dot(xmol.intor("int1e_spnucsp"))

    X = x2c_xmatrix(T, V, W, S, c_light)
    H_x2c = x2c_hcore_fw(T, V, W, S, X, c_light)

    if xuncontract and contr_coeff is not None:
        C = block_diag_spin(contr_coeff)
        H_x2c = reduce(np.dot, (C.T, H_x2c, C))

    return H_x2c, xmol


H_core_x2c, xmol = get_x2c_hcore(mol, xuncontract=XUNCONTRACT)
A_so = get_rmat(np.eye(nso), S_so)

helper_ref = x2c.SpinOrbitalX2CHelper(mol)
helper_ref.xuncontract = XUNCONTRACT
H_core_x2c_ref = helper_ref.get_hcore(mol)

print(f"xmol nao = {xmol.nao_nr()}, final nso = {H_core_x2c.shape[0]}")
print(f"H_core_x2c dtype = {H_core_x2c.dtype}")
print(f"||H - H^dagger|| = {np.linalg.norm(H_core_x2c - H_core_x2c.conj().T):.3e}")
print(f"||Im(H)|| = {np.linalg.norm(H_core_x2c.imag):.3e}")
print(f"||H - H_PySCF|| = {np.linalg.norm(H_core_x2c - H_core_x2c_ref):.3e}")

xmol nao = 33, final nso = 38
H_core_x2c dtype = complex128
||H - H^dagger|| = 7.768e-15
||Im(H)|| = 2.728e-03
||H - H_PySCF|| = 1.412e-10


---

## 4 · GHF 二电子项

X2C1e-GHF 只替换一电子 Hamiltonian。这里和非相对论 GHF 一样，用普通 Coulomb 积分构造 spin-orbital ERI：

$$
(p\sigma,q\sigma|r\tau,s\tau) = (pq|rs)
$$

exchange 允许 alpha/beta off-diagonal density block 参与收缩，因此整个 Fock 矩阵仍是 complex。

In [3]:
def build_spinorb_eri(eri):
    nao = eri.shape[0]
    nso = 2 * nao
    eri_so = np.zeros((nso, nso, nso, nso), dtype=np.complex128)
    spin_blocks = (slice(0, nao), slice(nao, nso))

    for same_spin_1 in spin_blocks:
        for same_spin_2 in spin_blocks:
            eri_so[same_spin_1, same_spin_1, same_spin_2, same_spin_2] = eri
    return eri_so


eri_so = build_spinorb_eri(eri)


def get_occ(mo_energy, nocc):
    e_idx = np.argsort(mo_energy)
    mo_occ = np.zeros_like(mo_energy)
    mo_occ[e_idx[:nocc]] = 1
    return mo_occ


def make_rdm1(mo_coeff, mo_occ):
    mocc = mo_coeff[:, mo_occ > 0]
    occ = mo_occ[mo_occ > 0]
    return (mocc * occ).dot(mocc.conj().T)


def get_jk(eri_so, dm):
    vj = np.einsum("pqrs,rs->pq", eri_so, dm, optimize=True)
    vk = np.einsum("prqs,rs->pq", eri_so, dm, optimize=True)
    return vj, vk


def get_veff(eri_so, dm):
    vj, vk = get_jk(eri_so, dm)
    return vj - vk


def get_fock(H_core, V_eff):
    return H_core + V_eff


def energy_elec(dm, H_core, V_eff):
    e1 = np.einsum("pq,qp->", H_core, dm, optimize=True)
    e2 = 0.5 * np.einsum("pq,qp->", V_eff, dm, optimize=True)
    return (e1 + e2).real, e2.real


def energy_tot(dm, H_core, V_eff):
    return E_nn + energy_elec(dm, H_core, V_eff)[0]


def get_init_guess(kind="minao"):
    if kind == "minao":
        # PySCF X2C-GHF also uses the GHF minao density by default.
        return scf.GHF(mol).init_guess_by_minao(mol).astype(np.complex128)

    if kind == "block":
        mo_energy, mo_coeff = eigh(H_core_nr, S_nr)
        e_idx = np.argsort(mo_energy)
        mo_alpha = mo_coeff[:, e_idx[:nalpha]]
        mo_beta = mo_coeff[:, e_idx[:nbeta]]
        dm_alpha = mo_alpha @ mo_alpha.conj().T
        dm_beta = mo_beta @ mo_beta.conj().T
        return block_diag(dm_alpha, dm_beta).astype(np.complex128)

    if kind == "hcore":
        mo_energy, mo_coeff = eigh(H_core_x2c, S_so)
        mo_occ = get_occ(mo_energy, nelec)
        return make_rdm1(mo_coeff, mo_occ)

    raise ValueError(f"unknown initial guess: {kind}")


print(f"eri_so shape = {eri_so.shape} ({eri_so.size:,} elements)")
print("X2C-GHF helper functions ready.")

eri_so shape = (38, 38, 38, 38) (2,085,136 elements)
X2C-GHF helper functions ready.


---

## 5 · DIIS

In [4]:
def get_err_vec(fock, dm, S, A):
    return A @ (fock @ dm @ S - S @ dm @ fock) @ A


def apply_diis(fock_list, err_vec_list):
    n = len(fock_list)
    dim = n + 1
    B = np.empty((dim, dim))
    B[-1, :] = -1
    B[:, -1] = -1
    B[-1, -1] = 0

    for i in range(n):
        for j in range(n):
            B[i, j] = np.vdot(err_vec_list[i], err_vec_list[j]).real

    rhs = np.zeros(dim)
    rhs[-1] = -1

    try:
        coeff = np.linalg.solve(B, rhs)[:-1]
    except np.linalg.LinAlgError:
        coeff = np.linalg.lstsq(B, rhs, rcond=None)[0][:-1]

    return np.einsum("i,ipq->pq", coeff, np.asarray(fock_list), optimize=True)


print("DIIS functions defined.")

DIIS functions defined.


---

## 6 · X2C-GHF + DIIS 完整运行

In [5]:
max_cycle = 100
max_diis = 8
conv_tol = 1e-9
conv_tol_dm = 1e-7
init_guess = "minao"

e_old = 0.0
fock_list = []
err_vec_list = []

dm = get_init_guess(init_guess)

print(f"Initial guess = {init_guess}")
print(f"{'Iter':>4s}  {'E_total':>16s}  {'dE':>10s}  {'dD':>10s}")
print("-" * 46)

converged = False
for cycle in range(max_cycle):
    V_eff = get_veff(eri_so, dm)
    fock = get_fock(H_core_x2c, V_eff)
    e_tot = energy_tot(dm, H_core_x2c, V_eff)

    err = get_err_vec(fock, dm, S_so, A_so)
    fock_list.append(fock)
    err_vec_list.append(err)
    if len(fock_list) > max_diis:
        fock_list.pop(0)
        err_vec_list.pop(0)

    if cycle >= 2:
        fock = apply_diis(fock_list, err_vec_list)

    mo_energy, mo_coeff = eigh(fock, S_so)
    mo_occ = get_occ(mo_energy, nelec)
    dm_new = make_rdm1(mo_coeff, mo_occ)

    e_diff = abs(e_tot - e_old)
    d_norm = np.linalg.norm(dm_new - dm)

    print(f"{cycle+1:4d}  {e_tot:16.10f}  {e_diff:10.3e}  {d_norm:10.3e}")

    if e_diff < conv_tol and d_norm < conv_tol_dm:
        converged = True
        dm = dm_new
        V_eff = get_veff(eri_so, dm)
        fock = get_fock(H_core_x2c, V_eff)
        e_tot = energy_tot(dm, H_core_x2c, V_eff)
        print(f"\nConverged in {cycle+1} iterations!")
        print(f"E_total (X2C-GHF+DIIS/cc-pVDZ) = {e_tot:.10f} Hartree")
        print(f"E_elec = {e_tot - E_nn:.10f}")
        print(f"E_nn   = {E_nn:.10f}")
        break

    dm = dm_new
    e_old = e_tot

if not converged:
    print("WARNING: X2C-GHF SCF did not converge!")

Initial guess = minao
Iter           E_total          dE          dD
----------------------------------------------
   1    -75.0925996685   7.509e+01   8.517e-01
   2    -75.4277035845   3.351e-01   1.428e-01
   3    -75.4417656086   1.406e-02   3.114e-02
   4    -75.4425792630   8.137e-04   1.267e-02
   5    -75.4427286270   1.494e-04   5.500e-03
   6    -75.4427554725   2.685e-05   3.814e-03
   7    -75.4427638678   8.395e-06   2.484e-03
   8    -75.4427665472   2.679e-06   1.292e-03
   9    -75.4427676362   1.089e-06   9.937e-04
  10    -75.4427685278   8.917e-07   1.513e-03
  11    -75.4427698799   1.352e-06   1.645e-02
  12    -75.4427839551   1.408e-05   8.436e-02
  13    -75.4428357836   5.183e-05   2.250e-02
  14    -75.4428400130   4.229e-06   4.042e-03
  15    -75.4428393549   6.580e-07   1.844e-02
  16    -75.4428436044   4.249e-06   1.589e-03
  17    -75.4428486172   5.013e-06   4.993e-02
  18    -75.4428609990   1.238e-05   9.488e-03
  19    -75.4428632617   2.263e-06   2

---

## 7 · 与 PySCF 对照

In [6]:
mf_ref = scf.GHF(mol).x2c1e()
mf_ref.with_x2c.xuncontract = XUNCONTRACT
mf_ref.conv_tol = 1e-10
mf_ref.verbose = 0
mf_ref.kernel()

spin_flip_norm = np.linalg.norm(dm[:nao, nao:]) + np.linalg.norm(dm[nao:, :nao])

print(f"\nPySCF X2C-GHF:     {mf_ref.e_tot:.10f} Hartree")
print(f"Our X2C-GHF+DIIS:   {e_tot:.10f} Hartree")
print(f"Difference:         {abs(mf_ref.e_tot - e_tot):.2e} Hartree")
print(f"||D_alpha_beta|| + ||D_beta_alpha|| = {spin_flip_norm:.3e}")


PySCF X2C-GHF:     -75.4428657635 Hartree
Our X2C-GHF+DIIS:   -75.4428657635 Hartree
Difference:         8.53e-13 Hartree
||D_alpha_beta|| + ||D_beta_alpha|| = 6.762e-01


---

## 8 · 总结

| 模块 | X2C-GHF 形式 |
|------|--------------|
| AO 基 | 与 GHF 一样的 alpha-all / beta-all spin-orbital AO |
| 一电子项 | $H_{\mathrm{core}}$ 替换为 complex Hermitian $H_{\mathrm{X2C}}$ |
| X2C 关键矩阵 | $X=C_SC_L^{-1}$，$R^\dagger\tilde{S}R=S$ |
| 二电子项 | 与 GHF 相同的 Coulomb/exchange spin block 收缩 |
| SCF | 对 complex generalized eigenvalue problem $FC=SC\epsilon$ 做 DIIS 迭代 |
| PySCF 对应 | `scf.GHF(mol).x2c1e()` / `SpinOrbitalX2CHelper` |